In [1]:
import paramiko
import stat
from pathlib import PurePosixPath
import os

In [2]:
HOST = os.environ.get("SFTP_HOST")
USER = os.environ.get("SFTP_USER")
PASSWORD = os.environ.get("SFTP_PASSWORD")
PORT = os.environ.get("SFTP_PORT")

#del os.environ["SSH_AUTH_SOCK"]

In [3]:
#import logging
#logging.basicConfig(level=logging.DEBUG)
#paramiko.util.log_to_file("paramiko.log")

In [7]:
def sftp_client(host=HOST, username=USER, port=PORT, password=PASSWORD):
    client = paramiko.SSHClient()
    client.set_missing_host_key_policy(paramiko.AutoAddPolicy())

    old_sock = os.environ.pop("SSH_AUTH_SOCK", None)
    connect_kwargs = {
        "username": username,
        "password": password,
        "timeout": 30,
        "allow_agent": False,
        "look_for_keys": False,   # <-- critical: stops ~/.ssh scan
    }
    if port:
        connect_kwargs["port"] = port

    client.connect(host, **connect_kwargs)
    connection = client.open_sftp()

    if old_sock:
        os.environ["SSH_AUTH_SOCK"] = old_sock

    return connection

In [8]:
sftp = sftp_client()

In [9]:
entries = sftp.listdir_attr("/")

In [10]:
def sftp_walk(sftp, root="/", max_depth=10):
    inventory = []

    def _walk(path, depth):
        if depth > max_depth:
            return
        try:
            entries = sftp.listdir_attr(path)
        except IOError as e:
            print(f"Skipping {path}: {e}")
            return
        for entry in entries:
            full_path = f"{path.rstrip('/')}/{entry.filename}"
            is_dir = stat.S_ISDIR(entry.st_mode)
            inventory.append({
                "path": full_path,
                "type": "dir" if is_dir else "file",
                "size": entry.st_size,
                "mtime": entry.st_mtime,
            })
            if is_dir:
                _walk(full_path, depth + 1)

    _walk(root, 0)
    return inventory

In [11]:
inv = sftp_walk(sftp, max_depth=2)

In [12]:
len(inv)

84596

In [13]:
inv[:10]

[{'path': '/.quarantine', 'type': 'dir', 'size': None, 'mtime': 1775123291},
 {'path': '/.tmb', 'type': 'dir', 'size': None, 'mtime': 1573220106},
 {'path': '/Aleksandrovka', 'type': 'dir', 'size': None, 'mtime': 1752861839},
 {'path': '/Aleksandrovka/12-2-167.pdf',
  'type': 'file',
  'size': 277905450,
  'mtime': 1744627287},
 {'path': '/Aleksandrovka/12-2-233.pdf',
  'type': 'file',
  'size': 257001276,
  'mtime': 1744627207},
 {'path': '/Aleksandrovka/12-2-269.pdf',
  'type': 'file',
  'size': 288342445,
  'mtime': 1744142601},
 {'path': '/Aleksandrovka/12-2-275.pdf',
  'type': 'file',
  'size': 193062022,
  'mtime': 1744627183},
 {'path': '/Aleksandrovka/12-2-280.pdf',
  'type': 'file',
  'size': 616239442,
  'mtime': 1743592635},
 {'path': '/Aleksandrovka/12-2-281.pdf',
  'type': 'file',
  'size': 536665675,
  'mtime': 1743763429},
 {'path': '/Aleksandrovka/12-2-57.pdf',
  'type': 'file',
  'size': 114752569,
  'mtime': 1744627251}]